# gpt3-local on Colab

Trains the from-scratch GPT-3-style model from
[github.com/yoonBot/gpt3-local](https://github.com/yoonBot/gpt3-local) using Colab's free GPU.
Training shows a live progress bar and a loss curve that updates in real time as it trains.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU` (free tier) or a paid tier for more VRAM.

**Colab sessions disconnect** after ~90 min idle or ~12h max, and the local disk is wiped each time.
Mount Google Drive (cell below) so checkpoints survive a disconnect and you can resume training.

## 1. Check the assigned GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/yoonBot/gpt3-local.git
%cd gpt3-local
!pip install -q -r requirements.txt

## 3. (Recommended) Mount Google Drive for persistent checkpoints

Colab's local disk is wiped on disconnect. Point `out_dir` at a Drive folder so training
can resume after a session drops, instead of restarting from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/gpt3-local-out'
os.makedirs(OUT_DIR, exist_ok=True)

## 4. Prepare data

Quick smoke test with tiny-shakespeare (seconds to tokenize). Swap in
`data/prepare_openwebtext.py` for a real pretraining run (much larger, budget real time/disk).

In [ ]:
!python data/prepare_shakespeare.py

## 5. Training helper

`train.py` is run **in-process** here (`import train; train.main()`) instead of via `!python train.py`.
That's what lets the loss-curve plot below render live, updating in place as training runs — a
subprocess launched with `!` has no way to draw into this notebook's own output area, only this
kernel's own Python code can. Pass the same names as `train.py`'s `--flags` as keyword arguments.

(Running `python train.py ...` from a plain terminal still works fine outside a notebook — it just
saves `out_dir/loss_curve.png` to disk each update instead of rendering it live, since there's no
notebook frontend to draw into.)

In [ ]:
import sys, importlib

def run_train(**kwargs):
    argv = ['train.py']
    for k, v in kwargs.items():
        flag = f'--{k}'
        if isinstance(v, bool):
            if v:
                argv.append(flag)
        else:
            argv.extend([flag, str(v)])
    sys.argv = argv
    import train
    importlib.reload(train)
    train.main()

## 6. Train

`gpt3-small` (125M) comfortably fits a free-tier T4 (16GB). Bump to `gpt3-medium`/`gpt3-large`
on a paid A100 tier, adding `gradient_checkpointing=True` for `gpt3-large`/`gpt3-xl`.
See the repo's README VRAM table for guidance on other sizes.

Watch the progress bar and the loss curve update live as this cell runs.

In [ ]:
run_train(
    config='gpt3-small', data_dir='data/shakespeare', out_dir=OUT_DIR,
    block_size=512, batch_size=12, grad_accum_steps=4,
    max_iters=5000, eval_interval=250,
)

### Resuming after a disconnect

Re-run the clone/install/mount/helper cells above, then rerun with `resume=True` (same `out_dir`,
pointed at the same Drive path) — it'll pick back up from the last saved checkpoint.

In [ ]:
run_train(
    config='gpt3-small', data_dir='data/shakespeare', out_dir=OUT_DIR,
    block_size=512, batch_size=12, grad_accum_steps=4,
    max_iters=5000, eval_interval=250, resume=True,
)

## 7. Generate text

In [ ]:
!python sample.py --ckpt "$OUT_DIR/ckpt.pt" --prompt "The meaning of life is" --max_new_tokens 200

## 8. (Optional) Retrieval-augmented generation

Upload some `.txt` files to a folder (e.g. via the Colab file browser, or
`from google.colab import files; files.upload()`), then index and query them.

In [ ]:
!python rag/build_index.py --docs_dir mydocs/ --out_dir rag_index/
!python rag/generate_with_rag.py --ckpt "$OUT_DIR/ckpt.pt" --index_dir rag_index/ \
    --query "your question here"

## 9. (Optional) Fine-tune calculator tool-use

Teaches the model to emit `<calc>expr</calc>` and splices in a real (safely evaluated) result
instead of a guessed one. Needs a vocab resize since the extended tokenizer
(`<calc>`/`</calc>` plus the chat turn tokens used below) adds 5 tokens to GPT-2's BPE vocab.

In [ ]:
!python tools/prepare_calc_data.py --n_examples 200000
!python tools/resize_embeddings.py --in_ckpt "$OUT_DIR/ckpt.pt" --out_ckpt "$OUT_DIR/ckpt_tool.pt"

run_train(
    config='gpt3-small', vocab_size=50262, data_dir='data/calc',
    out_dir=OUT_DIR + '_calc', init_from_ckpt=OUT_DIR + '/ckpt_tool.pt', max_iters=5000,
)

**Hit a `vocab_size` mismatch error here?** This repo's tokenizer has gained new special tokens
over time (e.g. the chat turn tokens added after this calculator step was first written), so a
checkpoint saved before that change can mismatch a freshly re-cloned copy of the repo — which
happens after a Colab disconnect wipes the local disk and the clone cell re-runs. It's fixable in
place, and the error message now tells you the exact command; it looks like:
```
python tools/resize_embeddings.py --in_ckpt out_calc/ckpt.pt --out_ckpt out_calc/ckpt.pt
```
This only adds fresh rows for the new tokens — everything the calculator fine-tune already learned
is preserved unchanged, so no re-training needed. Then retry the cell below.

In [ ]:
!python tools/generate_with_calc.py --ckpt "$OUT_DIR"_calc/ckpt.pt \
    --prompt "Q: What is 47 + 89?\nA:"

## 10. (Optional) Chat interface, built on the calculator checkpoint

A raw next-token model doesn't inherently know how to hold a conversation — this continues
fine-tuning from **`out_calc/ckpt.pt`** (section 9's calculator checkpoint) rather than starting
over from the plain resized checkpoint, so the chat model keeps the calculator behavior it already
learned instead of re-learning it from scratch. No `resize_embeddings.py` step is needed here —
`out_calc/ckpt.pt` already has the extended vocab (fine-tuning it again would raise
"already >= vocab; nothing to resize").

**You don't need to manually upload anything** if you're using the same Google Drive across
sessions: `out_calc/ckpt.pt` already lives at `{OUT_DIR}_calc/ckpt.pt` in your mounted Drive from
when you ran section 9, and mounting Drive again in a new session gives you access to that exact
file automatically. You'd only need to manually upload a checkpoint if it came from somewhere Drive
can't see (a different Google account, or trained on your own local machine) — in that case, use the
Colab file browser (left sidebar) or `from google.colab import files; files.upload()`, then point
`init_from_ckpt` below at wherever it lands.

(Run section 9 first if you haven't — this cell expects `out_calc/ckpt.pt` to already exist.)

`chat/prepare_chat_synthetic.py` alone only teaches turn-taking format + calculator delegation (a
few hundred templated patterns) — for a model that's actually broadly conversational, also run
`chat/prepare_chat_alpaca.py` (~52K real instruction-following examples).

In [ ]:
!python chat/prepare_chat_synthetic.py --n_examples 100000
!python chat/prepare_chat_alpaca.py

run_train(
    config='gpt3-small', vocab_size=50262, data_dir='data/chat_alpaca',
    out_dir=OUT_DIR + '_chat', init_from_ckpt=OUT_DIR + '_calc/ckpt.pt', max_iters=5000,
)

`--share` gets a public `gradio.live` URL, since Colab has no direct access to `localhost` —
click the link that's printed below to open the chat UI in a new tab. Add `--rag_index_dir rag_index/`
(from step 8) to answer from your own documents.

In [ ]:
!python chat/gradio_app.py --ckpt "$OUT_DIR"_chat/ckpt.pt --share